# **1. Perkenalan Dataset**

Dataset yang digunakan pada eksperimen ini adalah **Indonesian Job Classification Dataset**, yaitu kumpulan deskripsi lowongan pekerjaan berbahasa Indonesia yang terdiri dari **1.000 data** dengan **8 kategori pekerjaan** yang seimbang (masing-masing 125 data).

**Kategori pekerjaan:**
1. Education
2. Engineering
3. Finance & Accounting
4. Healthcare
5. Human Resources
6. IT & Software
7. Marketing & Sales
8. Operations & Supply Chain

**Struktur dataset (`jobs_raw.csv`):**
- `job_title` : Judul posisi pekerjaan
- `text` : Deskripsi pekerjaan dalam Bahasa Indonesia
- `category` : Label kategori pekerjaan (string)
- `label` : Label numerik (0–7)

Tujuan eksperimen ini adalah membangun pipeline preprocessing teks untuk mempersiapkan data sebelum digunakan pada proses pelatihan model klasifikasi.

# **2. Import Library**

Pada tahap ini, kita mengimpor pustaka Python yang dibutuhkan untuk analisis data, visualisasi, dan preprocessing teks.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

print("Libraries imported successfully.")

# **3. Memuat Dataset**

Pada tahap ini, kita memuat dataset mentah (`jobs_raw.csv`) ke dalam DataFrame, kemudian memeriksa struktur dan kualitas awal data.

In [ ]:
df = pd.read_csv('../jobs_raw.csv')
print(f"Shape dataset: {df.shape}")
print(f"Kolom: {df.columns.tolist()}")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

# **4. Exploratory Data Analysis (EDA)**

Pada tahap ini, kita melakukan eksplorasi untuk memahami karakteristik dataset sebelum preprocessing.

In [ ]:
# Distribusi kategori pekerjaan
plt.figure(figsize=(10, 5))
df['category'].value_counts().plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Distribusi Kategori Pekerjaan', fontsize=14)
plt.xlabel('Kategori')
plt.ylabel('Jumlah Data')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()
print(df['category'].value_counts())

In [ ]:
# Cek missing values
print("Missing values per kolom:")
print(df.isnull().sum())

In [ ]:
# Cek duplikasi
print(f"Jumlah data duplikat: {df.duplicated().sum()}")

In [ ]:
# Distribusi panjang teks (jumlah kata)
df['word_count_raw'] = df['text'].str.split().str.len()
plt.figure(figsize=(8, 4))
df['word_count_raw'].hist(bins=30, color='teal', edgecolor='black')
plt.title('Distribusi Jumlah Kata dalam Teks')
plt.xlabel('Jumlah Kata')
plt.ylabel('Frekuensi')
plt.tight_layout()
plt.show()
print(df['word_count_raw'].describe())

In [ ]:
# Contoh teks per kategori
for cat in df['category'].unique():
    sample = df[df['category'] == cat]['text'].iloc[0]
    print(f"[{cat}]\n{sample[:200]}\n")

# **5. Data Preprocessing**

Pada tahap ini kita melakukan pembersihan dan transformasi teks sebelum digunakan pada pelatihan model. Langkah-langkah:
1. **Case folding** — ubah semua teks ke huruf kecil
2. **Pembersihan karakter** — hapus karakter non-alfabet dan spasi berlebih
3. **Stopword removal** — hapus kata-kata umum Bahasa Indonesia menggunakan NLTK
4. **Pembuatan kolom baru** — simpan hasil bersih ke kolom `text_clean` dan hitung `word_count`
5. **Label encoding** — tambahkan kolom `label` numerik
6. **Simpan hasil** — ekspor ke `jobs_preprocessing.csv`

In [ ]:
def preprocess_text(text):
    if pd.isna(text):
        return ""
    # Case folding
    text = text.lower()
    # Hapus karakter non-alfabet
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Normalisasi spasi
    text = re.sub(r'\s+', ' ', text).strip()
    # Stopword removal (Bahasa Indonesia)
    tokens = text.split()
    stop_words = set(stopwords.words('indonesian'))
    tokens = [t for t in tokens if t not in stop_words]
    return ' '.join(tokens)

print("Fungsi preprocessing berhasil didefinisikan.")

In [ ]:
# Terapkan preprocessing ke kolom teks
df['text_clean'] = df['text'].apply(preprocess_text)
df['word_count'] = df['text_clean'].str.split().str.len()
print("Preprocessing selesai.")
df[['text', 'text_clean', 'word_count']].head(3)

In [ ]:
# Label encoding
label_mapping = {
    'Education': 0,
    'Engineering': 1,
    'Finance & Accounting': 2,
    'Healthcare': 3,
    'Human Resources': 4,
    'IT & Software': 5,
    'Marketing & Sales': 6,
    'Operations & Supply Chain': 7
}
df['label'] = df['category'].map(label_mapping)
print("Label mapping:")
for k, v in label_mapping.items():
    print(f"  {v}: {k}")

In [ ]:
# Distribusi panjang teks setelah preprocessing
plt.figure(figsize=(8, 4))
df['word_count'].hist(bins=30, color='coral', edgecolor='black')
plt.title('Distribusi Jumlah Kata Setelah Preprocessing')
plt.xlabel('Jumlah Kata')
plt.ylabel('Frekuensi')
plt.tight_layout()
plt.show()
print(df['word_count'].describe())

In [ ]:
# Simpan hasil preprocessing
output_cols = ['job_title', 'text_clean', 'category', 'label', 'word_count']
df[output_cols].to_csv('jobs_preprocessing.csv', index=False)
print(f"Berhasil menyimpan {len(df)} baris ke jobs_preprocessing.csv")
df[output_cols].head()